# 1B 1-Layer DeepSeek-V4.1-Flash LoopMoE — Colab TPU v6e-1 Training

- **Backbone**: `L=1` physical layer, `d_model=2048`, `V=65,536` (tied), `Single-Pass mHC (4 streams, 10 Sinkhorn iters)`, `Engram (209.04M, 20.5% sparse)`, `MLA + YaRN (d_qk=96+32=128, d_v=128, Splash Attention)`, `256 routed (512) + 1 shared (512) MoE top-5` (`Tokamax Megablox ragged_dot`), `LoopMoE IterAdaLN`.
- **Data (All L3)**: `openbmb/Ultra-FineWeb-L3` (EN/ZH) + `openbmb/UltraData-Code` (`L3`) + `openbmb/UltraData-Math` (`L3`).
- **Curriculum**: 20B (`1K->2K->4K->8K->16K->32K`, `R=1`) -> 1B (`4K`, `R=8`).
- **Optimizer**: Per-expert 3D `vmap` Newton-Schulz **Muon** (`ns_steps=4`) + BF16 **AdamW** (**Warmup-Stable, NO Decay**).
- **Checkpoints**: Saved locally (`CKPT_DIR`, e.g., Colab local disk or mounted Google Drive).

In [ ]:
# 1. Clone/update repository and install latest pinned stack from requirements.txt
import os
REPO_DIR = '/content/M3'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/GarD0m/M3.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q -U -r {REPO_DIR}/requirements.txt

In [ ]:
# 2. Verify TPU v6e-1 runtime in a subprocess so the Jupyter kernel does NOT lock libtpu.so
!python -c "import jax; print('Backend:', jax.default_backend()); print('Devices:', jax.devices()); [print(d, d.memory_stats()) for d in jax.local_devices()]"

In [ ]:
# 3. Configure local paths (optional: mount Google Drive for persistent checkpoints & JAX cache)
TOKENIZER_PATH = f'{REPO_DIR}/artifacts/tokenizer_64k/tokenizer.json'
CKPT_DIR = f'{REPO_DIR}/checkpoints'
CACHE_DIR = f'{REPO_DIR}/jax_cache'

# Optional: persist local checkpoints & compilation cache to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# CKPT_DIR = '/content/drive/MyDrive/loopmoe_checkpoints'
# CACHE_DIR = '/content/drive/MyDrive/loopmoe_jax_cache'

In [ ]:
# 4. Quick TPU v6e-1 MFU & HBM sanity benchmark (synthetic tokens, 5 steps)
%cd {REPO_DIR}
!python scripts/train_tpu.py \
    --data-mode synthetic \
    --ckpt-dir {CKPT_DIR} \
    --cache-dir {CACHE_DIR} \
    --start-stage 1 --end-stage 1 \
    --max-steps-per-stage 5 \
    --log-every 1

In [ ]:
# 5. Full 7-Stage L3 Curriculum Training (saves & auto-resumes checkpoints locally; expands R=1 -> R=8 at Stage 7)
%cd {REPO_DIR}
!python scripts/train_tpu.py \
    --data-mode stream \
    --tokenizer {TOKENIZER_PATH} \
    --ckpt-dir {CKPT_DIR} \
    --cache-dir {CACHE_DIR} \
    --keep-last-k 3 \
    --start-stage 1 --end-stage 7 \
    --log-every 10